# Домашнее задание 9: авторегрессия против диффузии

**Цель:** обучить на одних и тех же данных (MNIST) две генеративные модели из лекции — авторегрессионную ImageGPT и диффузионную DDPM, — ускорить диффузию с помощью DDIM и сравнить всё по метрикам FID и precision/recall.

**Где запускать:** всё рассчитано на **Google Colab с GPU T4** (Runtime → Change runtime type → T4 GPU). Ориентиры по времени на T4 приведены в каждом задании.

Теория — в `9-1 Metrics + VAR.ipynb` (метрики, авторегрессия) и `9-2 Diffusion.ipynb` (DDPM, DDIM).

## Система оценки (макс. 10 баллов)

| задание | баллы |
|---|---|
| 1. ImageGPT | 4 |
| 2. DDPM и DDIM | 4 |
| 3. Метрики: сравниваем модели | 2 |

**+1 балл** может добавить ревьюер за интересный анализ, визуализацию или нестандартное решение.

Требования ко всем заданиям:
- код должен запускаться сверху вниз без ошибок;
- в каждом задании — **график / картинки / таблица с результатами** и **текстовые выводы** (2–5 предложений): что получилось и почему;
- баллы снижаются за отсутствие выводов, даже если код работает.

In [ ]:
# !pip install -q diffusers scipy "torchmetrics[image]"

import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED =  # YOUR SEED HERE
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Задание 1. ImageGPT (4 балла)

Реализуйте модель ImageGPT ([статья](https://cdn.openai.com/papers/Generative_Pretraining_from_Pixels_V2.pdf), [блогпост](https://openai.com/index/image-gpt/)) — авторегрессию по пикселям бинаризованного MNIST 32×32: картинка вытягивается в последовательность в растровом порядке, каждый следующий пиксель предсказывается трансформером по предыдущим.

Код с датасетом и скелеты классов подготовили для вас. Нужно:
1. Завершить код архитектуры: маска внимания, decoder-блок, модель целиком.
2. Самостоятельно реализовать обучение: train loop, график лосса, генерацию картинок.

**Оценивание:**
- 2.5 балла — архитектура реализована правильно (маска, блок, forward и генерация);
- 1.5 балла — модель сошлась: график лосса и сетка сгенерированных цифр, узнаваемых на глаз.

In [ ]:
# попробуйте визуализировать датасет, понять, что у вас за данные
import torch
import torchvision
from torchvision.datasets import MNIST


root = "./.data"
train_dataset = MNIST(root=root, train=True, download=True)
test_dataset = MNIST(root=root, train=False, download=True)
    
transform = torchvision.transforms.Resize((32, 32))
train_data = torch.stack([transform(img.unsqueeze(0)) for img in train_dataset.data]).numpy().transpose(0, 2, 3, 1)
test_data = torch.stack([transform(img.unsqueeze(0)) for img in test_dataset.data]).numpy().transpose(0, 2, 3, 1)

train_data = (train_data > 128).astype("int64")
test_data = (test_data > 128).astype("int64")

train_data = np.transpose(train_data, (0, 3, 1, 2))
test_data = np.transpose(test_data, (0, 3, 1, 2))

In [ ]:
class MultiheadAttention(nn.MultiheadAttention):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__(embed_dim, num_heads)

    def get_attention_mask(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        # define attention mask, it should contain
        # - zeros under and on the main diagonal
        # - minus Inf above the main diagonal

        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_mask = self.get_attention_mask(x)
        return super().forward(x, x, x, attn_mask=attn_mask, need_weights=False)[0]


def test_attention_mask() -> None:
    x = torch.zeros(2, 4, 16)  # (pixel_num, batch_size, emb_dim)
    mask = np.array([[0.0, -np.inf], [0.0, 0.0]])
    layer = MultiheadAttention(16, 8)
    attention_mask = layer.get_attention_mask(x)
    assert attention_mask.size() == (x.size(0), x.size(0))
    assert np.allclose(attention_mask.numpy(), mask)
    out = layer(x)
    assert x.size() == out.size()


test_attention_mask()

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        """
        :param embed_dim: dimension of embedding space
        :param num_heads: number of attention heads
        """
        super().__init__()
        assert embed_dim % num_heads == 0

        # YOUR CODE HERE
        # define multihead attention
        # define LayerNorm 
        # define MLP - 2 linear layers with ReLU
        raise NotImplementedError()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError()


def test_decoder_block() -> None:
    block = DecoderBlock(embed_dim=12, num_heads=4)
    x = torch.zeros(4, 28, 12)
    assert x.shape == block(x).shape


test_decoder_block()

In [ ]:
class ImageGPT(nn.Module):
    def __init__(
        self, input_shape: tuple[int, int], embed_dim: int, num_heads: int, num_layers: int
    ) -> None:
        super().__init__()

        self.embed_dim = embed_dim
        self.input_shape = input_shape
        self.criterion = nn.BCEWithLogitsLoss()

        # "start of sequence" token (we initialize it from Normal distribution)
        self.sos = torch.nn.Parameter(torch.zeros(embed_dim))
        nn.init.normal_(self.sos)
        # YOUR CODE HERE
        # 1) define token_embeddings
        # 2) define position_embeddings (they will be learnable)
        # (use torch.nn.Embedding)

        self.layers = nn.ModuleList()
        # 1) add decoder blocks to self.layers list
        # 2) define last LayerNorm
        # 3) define final Linear layer (without bias)
        raise NotImplementedError()

    def add_sos_token(self, embeddings: torch.Tensor) -> torch.Tensor:
        batch_size = embeddings.size(1)
        # YOUR CODE HERE
        # prepend sos (start of sequence) token
        # 1) repeat sos token batch_size times (make it of size (1, batch_size, emd_size))
        # 2) drop last embedding from embeddings
        # 3) concat repeated sos token to embeddings (after dropping)
        raise NotImplementedError()
        return embeddings

    def add_pos_embeddings(self, embeddings: torch.Tensor) -> torch.Tensor:
        length = embeddings.size(0)
        # YOUR CODE HERE
        # add positional embeddings
        # 1) define tensor with positions (just torch.arange) of size (length, 1)
        # 2) add position embeddings to initial embeddings
        raise NotImplementedError()

        return embeddings

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.long()
        x = x.reshape(x.size(0), -1)  # (batch_size, length)
        x = x.permute(1, 0)

        embeddings = self.token_embeddings(x)  # (length, batch_size, emb_size)
        embeddings = self.add_sos_token(embeddings)
        embeddings = self.add_pos_embeddings(embeddings)
        # YOUR CODE HERE
        # 1) apply all decoder layers
        # 2) apply final LayerNorm and Linear layer
        raise NotImplementedError()

        return logits.permute(1, 0, 2)  # (length, batch_size, emb_size) -> (batch_size, length, emb_size)

    def loss(self, x: torch.Tensor) -> dict:
        logits = self(x)
        loss = self.criterion(logits.reshape(-1), x.reshape(-1).float())
        return {"total_loss": loss}

    @torch.no_grad()
    def sample(self, n_samples: int) -> np.ndarray:
        # YOUR CODE HERE
        # hint: You can use PixelCNN sample from lecture for inspiration
        # 1. You sample not an image (2d) but a sequence (1d) that will be reshaped in the end
        # 2. Instead multinomial you can use Bernoulli distribution
        raise NotImplementedError()


def test_image_gpt() -> None:
    image_gpt = ImageGPT(input_shape=(2, 2), embed_dim=12, num_heads=4, num_layers=2)
    x = torch.LongTensor([[0, 1, 0, 0], [0, 1, 1, 1]])
    assert image_gpt(x).shape == torch.Size([2, 4, 1])
    assert image_gpt.loss(x)["total_loss"].requires_grad == True
    assert image_gpt.sample(1).shape == torch.Size([1, 1, 2, 2])


test_image_gpt()

In [ ]:
# ОБУЧЕНИЕ: teacher forcing — на вход вся настоящая картинка, маска не даёт подсмотреть будущее,
#           сеть за один проход предсказывает P(пиксель = 1) для каждого пикселя по предыдущим; лосс — BCE.
# ИНФЕРЕНС: пиксели генерируются по одному в растровом порядке (32·32 = 1024 прохода сети):
#           предсказали распределение следующего пикселя → сэмплировали значение → дописали к последовательности.

# YOUR CODE HERE
# обучение, график лосса, генерация

## Задание 2. DDPM и DDIM (4 балла)

Обучите на том же MNIST диффузионную модель из лекции `9-2` (разделы 3–4). Сеть (U-Net) и данные даём готовыми — реализуйте сам процесс диффузии **без** `diffusers.DDPMScheduler`: только формулы из лекции и `torch`. Модель безусловная: на вход только $x_t$ и $t$.

1. **Schedule и forward process.** Линейный $\beta_t$ от $10^{-4}$ до $0.02$, $T = 1000$; посчитайте $\alpha_t$, $\bar\alpha_t$, $\bar\alpha_{t-1}$ и $\tilde\beta_t$. Функция `q_sample(x0, t, noise)`: $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$.
2. **Обучение.** Функция `diffusion_loss` — $L_\text{simple} = \|\epsilon - \epsilon_\theta(x_t, t)\|^2$ на одном батче со случайным $t$ для каждой картинки; цикл обучения `train` дан готовым. Ориентир — 10–15 эпох. График лосса.
3. **DDPM-сэмплер.** Функция `ddpm_sample(model, n, x_T=None)`: $T$ шагов $x_{t-1} = \mu_\theta(x_t, t) + \sigma_t z$, $\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta\big)$, $\sigma_t^2 = \tilde\beta_t$, на последнем шаге без шума. Сетка 8×8 сэмплов.
4. **DDIM-сэмплер.** Функция `ddim_sample(model, num_steps, x_T)` с $\sigma = 0$: возьмите `num_steps` равномерно расставленных timestep'ов от $T-1$ до 0; на каждом шаге оцените $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta)/\sqrt{\bar\alpha_t}$ и перейдите на следующий уровень $s$: $x_s = \sqrt{\bar\alpha_s}\,\hat x_0 + \sqrt{1-\bar\alpha_s}\,\epsilon_\theta$. Сравните **с одним и тем же** $x_T$ (8 сэмплов): DDPM 1000 шагов, DDIM 50 и DDIM 20 — время и картинки рядом.

**Оценивание:**
- 0.5 балла — schedule и `q_sample`;
- 1.5 балла — корректный `diffusion_loss` (случайный $t$ для каждой картинки, MSE по шуму) и сходящийся лосс;
- 1 балл — DDPM-сэмплер и сетка цифр, узнаваемых на глаз;
- 1 балл — DDIM-сэмплер, сравнение с DDPM и ответ: почему DDIM 50 и DDIM 20 дают одни и те же цифры, а DDPM с тем же $x_T$ — другие.

In [ ]:
# Данные для диффузии подготовили для вас: MNIST 32×32 в градациях серого, нормированный в [-1, 1]
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

root = "./.data"
mnist_train = MNIST(root=root, train=True, download=True)
mnist_test = MNIST(root=root, train=False, download=True)


def to_diffusion_tensor(ds: MNIST) -> tuple[torch.Tensor, torch.Tensor]:
    x = ds.data.float().unsqueeze(1) / 255.0                   # (N, 1, 28, 28) в [0, 1]
    x = F.interpolate(x, size=(32, 32), mode="bilinear", align_corners=False)
    return x * 2 - 1, ds.targets.clone()                       # [-1, 1]


x_train, y_train = to_diffusion_tensor(mnist_train)
x_test, y_test = to_diffusion_tensor(mnist_test)
diffusion_loader = DataLoader(TensorDataset(x_train), batch_size=128, shuffle=True, drop_last=True)  # батч: (x0,)
print(x_train.shape, x_train.min().item(), x_train.max().item())

In [ ]:
# Готовая сеть: безусловный U-Net из diffusers
from diffusers import UNet2DModel

unet = UNet2DModel(
    sample_size=32,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 128, 128),
    down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
).to(device)

print(f"params: {sum(p.numel() for p in unet.parameters()) / 1e6:.1f}M")
# использование: eps_pred = unet(x_t, t).sample, где t — LongTensor (B,)

In [ ]:
# ПОДГОТОВКА: schedule — фиксированные числа, не обучаются. Нужны и в обучении (q_sample), и в инференсе (сэмплеры).
# q_sample используется ТОЛЬКО в обучении: зашумляет x0 сразу до шага t одной формулой.

# 1. Schedule и forward process
T = 1000

# основные параментры
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
alpha_bars_prev = torch.cat([torch.ones(1, device=device), alpha_bars[:-1]])
beta_tildes = (1.0 - alpha_bars_prev) / (1.0 - alpha_bars) * betas


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    """
    :param x0: чистые картинки (B, 1, 32, 32)
    :param t: timestep'ы, LongTensor (B,)
    :param noise: шум той же формы, что x0
    :return: x_t
    """
    # YOUR CODE HERE
    ...


# дано: функция для отрисовки сетки картинок из [-1, 1]
def show_grid(images: torch.Tensor, nrow: int = 8, title: str = "") -> None:
    grid = torchvision.utils.make_grid((images.cpu() + 1) / 2, nrow=nrow, padding=2)
    plt.figure(figsize=(nrow, len(images) / nrow + 0.5))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# ОБУЧЕНИЕ: на каждой итерации — батч чистых x0, для каждой картинки случайный t и шум ε,
#           x_t = q_sample(x0, t, ε), сеть по (x_t, t) предсказывает ε, лосс — MSE(ε_pred, ε).
#           Длинная цепочка шагов здесь не нужна: каждый пример — один шаг.

# 2. Обучение
def diffusion_loss(model: nn.Module, x0: torch.Tensor, alpha_bars: torch.Tensor) -> torch.Tensor:
    """
    L_simple на одном батче.

    :param x0: чистые картинки (B, 1, 32, 32)
    :param alpha_bars: alpha_bar_t из schedule, длины T
    :return: скалярный лосс

    q_sample берёт schedule из глобальных переменных; alpha_bars здесь нужен, чтобы знать T = len(alpha_bars)
    """
    # YOUR CODE HERE: случайный t, шум, x_t через q_sample, предсказание шума, MSE
    ...


def train(model: nn.Module, loader: DataLoader, alpha_bars: torch.Tensor,
          num_epochs: int = 10, lr: float = 2e-4) -> list[float]:
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    losses = []
    model.train()
    for epoch in range(num_epochs):
        for (x0,) in loader:
            loss = diffusion_loss(model, x0.to(device), alpha_bars)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses[-len(loader):]):.4f}")
    return losses


losses = train(unet, diffusion_loader, alpha_bars)

# YOUR CODE HERE: график лосса

### Инференс: генерация из шума

Обучение закончено: `unet` умеет по $x_t$ и $t$ угадывать шум. Ни один сэмплер в обучении не вызывался — там мы прыгали из $x_0$ в случайный $x_t$ через `q_sample` и считали лосс на одном шаге.

Чтобы получить **новую** картинку, идём в обратную сторону: стартуем с чистого шума $x_T \sim \mathcal N(0, I)$ и шаг за шагом убираем шум, на каждом шаге спрашивая у сети $\epsilon_\theta(x_t, t)$. Сеть одна и та же, а способов шагать два:

| | DDPM (пункт 3) | DDIM (пункт 4) |
|---|---|---|
| шаг | $x_t \to x_{t-1}$ | $x_t \to \hat x_0 \to x_s$, через много шагов сразу |
| шум на шаге | добавляем $\sigma_t z$ | не добавляем ($\sigma = 0$) |
| вызовов сети | 1000 | 20–50 |

Переобучать модель для DDIM не нужно: лосс $L_\text{simple}$ не зависит от того, каким сэмплером потом пользуются.

In [ ]:
# ИНФЕРЕНС (DDPM): старт с чистого шума x_T, 1000 шагов x_t → x_{t-1}.
#           На каждом шаге: сеть предсказывает ε → считаем μ_θ → добавляем свежий шум σ_t z (кроме последнего шага).
#           Обученная сеть та же, что в ячейке обучения; градиенты не нужны (torch.no_grad).

# 3. DDPM-сэмплер
@torch.no_grad()
def ddpm_sample(model: nn.Module, n: int, x_T: torch.Tensor | None = None) -> torch.Tensor:
    """
    :param n: число картинок
    :param x_T: стартовый шум (n, 1, 32, 32); если None — сэмплируется
    :return: картинки (n, 1, 32, 32) в [-1, 1]
    """
    model.eval()
    x = torch.randn(n, 1, 32, 32, device=device) if x_T is None else x_T.clone()
    for t in reversed(range(T)):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        # YOUR CODE HERE: eps_theta, mu_theta, шаг x_{t-1} (на t = 0 без шума)
        x = ...
    return x.clamp(-1, 1)


samples = ddpm_sample(unet, 64)
show_grid(samples, nrow=8, title="DDPM, 1000 шагов")

In [ ]:
# ИНФЕРЕНС (DDIM): та же обученная сеть, другой способ шагать. Старт с x_T, num_steps шагов (20–50) по редкой сетке t.
#           На каждом шаге: сеть предсказывает ε → оцениваем x̂_0 → прыгаем сразу на уровень s, шум не добавляем.
#           Поэтому результат зависит только от x_T.

# 4. DDIM-сэмплер
@torch.no_grad()
def ddim_sample(model: nn.Module, num_steps: int, x_T: torch.Tensor) -> torch.Tensor:
    """
    :param num_steps: число шагов
    :param x_T: стартовый шум (n, 1, 32, 32)
    :return: картинки (n, 1, 32, 32) в [-1, 1]
    """
    model.eval()
    x = x_T.clone()
    n = x.shape[0]
    timesteps = torch.linspace(T - 1, 0, num_steps).long().tolist()
    for i, t in enumerate(timesteps):
        t_batch = torch.full((n,), t, device=device, dtype=torch.long)
        alpha_bar_s = alpha_bars[timesteps[i + 1]] if i + 1 < num_steps else torch.tensor(1.0, device=device)  # после последнего шага \bar\alpha_s = 1, т.е. x_s = \hat x_0
        # YOUR CODE HERE: eps_theta, \hat x_0, переход на уровень s
        x = ...
    return x.clamp(-1, 1)


# дано: сравнение с одним и тем же x_T
x_T = torch.randn(8, 1, 32, 32, device=device)
samplers = {
    "DDPM 1000": lambda: ddpm_sample(unet, 8, x_T),
    "DDIM 50": lambda: ddim_sample(unet, 50, x_T),
    "DDIM 20": lambda: ddim_sample(unet, 20, x_T),
}
for name, sample_fn in samplers.items():
    start = time.time()
    out = sample_fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    show_grid(out, nrow=8, title=f"{name}: {time.time() - start:.1f} с")

# YOUR CODE HERE: выводы — почему DDIM 50 и DDIM 20 дают одни и те же цифры, а DDPM с тем же x_T — другие

## Задание 3. Метрики: сравниваем модели (2 балла)

Сравним все модели по метрикам из лекции `9-1`: FID и precision/recall.

Считаем **стандартный FID**: признаки Inception-v3 (2048, слой перед классификатором) — та же сеть и те же веса, что в статьях, через `torchmetrics`. Картинки 32×32 с одним каналом сеть сама растягивает до 299×299, канал копируем в три. Precision/recall считаем на тех же признаках. Функции метрик даём готовыми.

**Честное сравнение.** ImageGPT генерирует бинарные картинки, а диффузия — в градациях серого. Поэтому перед подсчётом признаков **все картинки бинаризуются** (порог 0 в масштабе $[-1, 1]$) — это уже сделано внутри `extract_features`.

**Оговорка.** 2000 картинок — мало для 2048-мерных признаков: FID смещён вверх и зависит от $N$ (лекция `9-1`). Сравнивать модели между собой при одинаковом $N$ можно, с числами из статей — нет.

Для каждой строки сгенерируйте по 2000 картинок и посчитайте FID, precision, recall относительно 2000 тестовых картинок:
- реальные train vs test (нижняя граница FID);
- ImageGPT;
- DDPM 1000 шагов;
- DDIM 20.

**Оценивание:**
- 1 балл — таблица по всем строкам;
- 1 балл — выводы: какая модель лучше по FID и совпадает ли это с визуальным впечатлением; во сколько раз DDIM быстрее DDPM и сколько качества это стоит.


In [ ]:
# Готовые функции метрик
from scipy import linalg
from torchmetrics.image.fid import FrechetInceptionDistance

# Inception-v3 со стандартными FID-весами (как в pytorch-fid / статьях); вход — uint8 (N, 3, H, W), ресайз до 299 внутри
inception = FrechetInceptionDistance(feature=2048).inception.to(device).eval()


def binarize(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x > 0, 1.0, -1.0)


@torch.no_grad()
def extract_features(images: torch.Tensor, batch_size: int = 200) -> np.ndarray:
    # images: (N, 1, 32, 32) в [-1, 1]; бинаризация — внутри
    feats = []
    for i in range(0, len(images), batch_size):
        x = binarize(images[i:i + batch_size])
        x = ((x + 1) * 127.5).round().to(torch.uint8).repeat(1, 3, 1, 1)   # [-1, 1] -> uint8, 1 -> 3 канала
        feats.append(inception(x.to(device)).cpu())
    return torch.cat(feats).numpy().astype(np.float64)


def fid(feats_real: np.ndarray, feats_fake: np.ndarray) -> float:
    mu_r, mu_f = feats_real.mean(0), feats_fake.mean(0)
    cov_r, cov_f = np.cov(feats_real, rowvar=False), np.cov(feats_fake, rowvar=False)
    covmean, _ = linalg.sqrtm(cov_r @ cov_f, disp=False)
    covmean = covmean.real
    return float(((mu_r - mu_f) ** 2).sum() + np.trace(cov_r + cov_f - 2 * covmean))


def precision_recall(feats_real: np.ndarray, feats_fake: np.ndarray, k: int = 3) -> tuple[float, float]:
    # k-NN оценка многообразий (Kynkäänniemi et al., 2019)
    real, fake = torch.from_numpy(feats_real).float(), torch.from_numpy(feats_fake).float()

    def radii(x: torch.Tensor) -> torch.Tensor:
        d = torch.cdist(x, x)
        return d.kthvalue(k + 1, dim=1).values          # +1: расстояние до самого себя = 0

    def coverage(points: torch.Tensor, manifold: torch.Tensor, r: torch.Tensor) -> float:
        d = torch.cdist(points, manifold)               # (N_points, N_manifold)
        return (d <= r[None, :]).any(dim=1).float().mean().item()

    precision = coverage(fake, real, radii(real))       # доля fake внутри многообразия real
    recall = coverage(real, fake, radii(fake))          # доля real внутри многообразия fake
    return precision, recall


feats_test = extract_features(x_test[:2000])

In [ ]:
# ИНФЕРЕНС + ОЦЕНКА: генерируем картинки уже обученными моделями (ImageGPT, DDPM, DDIM 20),
#           прогоняем их и реальные картинки через Inception-v3 и сравниваем распределения признаков (FID, precision, recall).
#           Ничего не обучаем.

# YOUR CODE HERE
# генерация для всех строк, FID / precision / recall, таблица